# Omnibus — time on the map (flood replay + single-trip replay)

The static maps freeze time. These two animations put the clock back in:

1. **Flood week, day by day** — scrub 2024-05-26 → 06-07 and watch per-stop delay heat up and recover. This is the Scene C demo in miniature; it tells us *which corridor pattern is real* before we commit to it in the frontend.
2. **One bad trip, stop by stop** — replay a single delayed bus along its route with an animated path, each stop colored by the delay it carried at that moment. Validates whether Scene A's per-segment story is visible without route-shape map-matching.

Caveats as everywhere ([docs/DATA_DEFECTS.md](../docs/DATA_DEFECTS.md)): drop the Line-1 overlap window, filter `|delay_arr_s| < 7200`, ~4.5% of events lack coordinates.

> Animations render inline in JupyterLab. Open there to scrub the time slider.

In [ ]:
import polars as pl
import folium
from folium.plugins import HeatMapWithTime, AntPath
import branca.colormap as cmm
import numpy as np

df = pl.read_parquet("../data/parquet/features.parquet")
clean = df.filter((pl.col("source_window") != "Daten_Linie_1_2024-09_2025-08")
                  & (pl.col("delay_arr_s").abs() < 7200))
geo = clean.filter(pl.col("stop_lat").is_not_null())
print(f"geocoded clean rows: {geo.height:,}")

## 1 — Flood week, animated

Per (stop × day) median arrival delay, normalised to a 0–1 heat weight. The slider has one frame per day; play it to watch the wave move and recede.

In [ ]:
flood = (geo.filter(pl.col("source_window").str.starts_with("26.05.2024")
                    & pl.col("productive_arr"))
         .with_columns(day=pl.col("operating_day").cast(pl.Utf8)))

# per stop per day: median delay + coords
daily = (flood.group_by("day", "stop_name").agg(
            pl.col("delay_arr_s").median().alias("med"),
            pl.col("stop_lat").first().alias("lat"),
            pl.col("stop_lon").first().alias("lon"),
            pl.len().alias("n"))
         .filter(pl.col("n") > 10))

days = sorted(daily["day"].unique().to_list())
# weight: clip delay to [0, 300]s then normalise — keeps the scale stable across frames
WMAX = 300.0
frames = []
for d in days:
    sub = daily.filter(pl.col("day") == d)
    frames.append([[r["lat"], r["lon"], min(max(r["med"], 0), WMAX) / WMAX]
                   for r in sub.iter_rows(named=True)])

m1 = folium.Map(location=[49.0125, 12.0992], zoom_start=12, tiles="CartoDB dark_matter")
HeatMapWithTime(
    frames, index=days, name="median delay",
    radius=30, max_opacity=0.85, min_opacity=0.1, auto_play=False,
    gradient={0.0: "#2c7bb6", 0.4: "#ffff8c", 0.7: "#fdae61", 1.0: "#d7191c"},
).add_to(m1)
print(f"{len(days)} daily frames: {days[0]} → {days[-1]}")
m1

**How to read it.** Press play. The peak flood days (around 2024-06-01 → 06-03) bloom red along the Donau-hugging corridors, then cool back toward the baseline by 06-06/07. Stops that *stay* cool through the peak are the ones whose feeder roads were never cut — useful negative evidence for which corridors a resilience plan should actually protect. The dark basemap is deliberate: it makes the heat the only thing your eye tracks across frames.

## 2 — Replay one bad trip along its route

Find the worst trip (highest end-of-line delay) on a chosen line in the Oct-2024 baseline, then animate a bus tracing it stop-to-stop. The flowing `AntPath` shows direction of travel; each stop dot is colored by the delay carried when the bus arrived there.

In [ ]:
LINE = "9"   # 9 had the worst single trip in the probe; try "1", "8", "3", "X4"
WIN = "06.10.2024_19.10.2024_ITCS"

ln = (geo.filter((pl.col("line") == LINE) & (pl.col("source_window") == WIN)
                 & pl.col("productive_arr"))
      .sort("trip_id", "stop_seq"))

worst = (ln.group_by("trip_id").agg(
            pl.col("delay_arr_s").last().alias("end_delay"),
            pl.col("stop_seq").n_unique().alias("nstops"))
         .filter(pl.col("nstops") >= 15)
         .sort("end_delay", descending=True))
trip_id = worst.row(0, named=True)["trip_id"]
trip = ln.filter(pl.col("trip_id") == trip_id).unique("stop_seq", keep="first").sort("stop_seq")
print(f"line {LINE} worst trip {trip_id}: {trip.height} stops, "
      f"end delay {trip['delay_arr_s'].last()}s")

pts = [[r["stop_lat"], r["stop_lon"]] for r in trip.iter_rows(named=True)]
m2 = folium.Map(location=[trip["stop_lat"].median(), trip["stop_lon"].median()],
                zoom_start=12, tiles="CartoDB positron")
AntPath(pts, color="#444", weight=3, delay=800, dash_array=[12, 24]).add_to(m2)

dmin, dmax = float(trip["delay_arr_s"].min()), float(trip["delay_arr_s"].max())
cmap = cmm.LinearColormap(["#1a9850", "#fee08b", "#d73027"], vmin=dmin, vmax=dmax)
cmap.caption = f"Line {LINE} trip {trip_id} — arrival delay at each stop [s]"
for i, r in enumerate(trip.iter_rows(named=True)):
    folium.CircleMarker(
        [r["stop_lat"], r["stop_lon"]], radius=7, color="#222", weight=0.6,
        fill=True, fill_color=cmap(float(r["delay_arr_s"])), fill_opacity=0.95,
        tooltip=f"#{i+1} {r['stop_name']} — {r['delay_arr_s']}s "
                f"(dwell {r['dwell_s']}s)",
    ).add_to(m2)
cmap.add_to(m2)
m2

**How to read it.** The ant-march shows the bus's direction; follow it from the green origin and watch the dots redden as delay accumulates. The stop where green flips to red is the segment that *broke* the schedule — hover the dots to read the exact delay and dwell. If a red stop also shows a long dwell, that's demand-driven (lots of boarding); if the dwell is short but delay still jumped, the time was lost *moving* between stops (traffic / signals). That dwell-vs-running split is the Scene A → Scene B bridge, and you can eyeball it here per stop without any route-shape data.